# Import Modules

In [1]:
import pandas as pd
import numpy as np
import datetime as dt

## Functions

In [2]:
# Enable import of other modules
import sys
ROOT = r"C:\Users\g.varvounis\Documents\RiC"
if ROOT not in sys.path:
    sys.path.append(ROOT)


from model.option_pricer import BachelierPricer

## Import Data

In [3]:
df = pd.read_excel(r"C:\Users\g.varvounis\Documents\RiC\_data\OptionInputs.xlsx")

In [4]:
df = df.fillna(method="ffill")

In [5]:
df["date"] = pd.to_datetime(df["date"])
df= df.set_index("date")
df.index.name=None
df.index = pd.to_datetime(df.index)

In [6]:
df.head(2)

,MCP,FWD,TTF MA,EUA Y0
2026-10-01 00:00:00,176.5050,NaN,75.503,85.09
2026-10-01 01:00:00,172.8675,NaN,75.503,85.09


In [7]:
df_calibration_data = pd.read_excel(
    r"C:\Users\g.varvounis\OneDrive - HELLENiQ ENERGY\Risk Section Activity\RiC Monitoring\Reporting.xlsx",
    sheet_name="Market CSS")[["Date", "Market CSS (Spot)"]]

In [8]:
df_calibration_data.columns = ["Date", "CSS"]
df_calibration_data = df_calibration_data.set_index("Date", drop=True)
df_calibration_data.index = pd.to_datetime(df_calibration_data.index)
df_calibration_data.index.name=None
df_calibration_data = df_calibration_data.dropna()
#df_calibration_data

# Product Parameters

In [9]:
ng_coef = 2.1384
eua_coef = 0.3858

price_cor_bool = True
monthly_vol=True
dicount_rate= 0.08
premium_per_option = 12.65
strike=0

#num_pers=252 # 1 hour spacing 8760
num_pers=12 # 1 hour spacing 8760, 1 day 252, 1 month 12
volume_per_hour_per_option_mw = 50

In [10]:
valuation_date = dt.datetime(2026, 11, 6, 23, 0, 0)

In [11]:
df_past = df.loc[df.index <= valuation_date]
df_future = df.loc[df.index > valuation_date]

In [12]:
df.columns

Index(['MCP', 'FWD', 'TTF MA', 'EUA Y0'], dtype='object')

## Calcualtions

In [13]:
pricer = BachelierPricer(
    heat_rate=ng_coef, emission_factor=eua_coef, strike_price=strike, premium_per_option=premium_per_option, mw_per_option_per_hour=volume_per_hour_per_option_mw)

In [14]:
# sigma_power = pricer.calculate_historical_abs_vol(prices=df_future["FWD"], periods_per_year=num_pers)
# sigma_ng = pricer.calculate_historical_abs_vol(prices=df_future["TTF MA"], periods_per_year=num_pers)
# sigma_eua = pricer.calculate_historical_abs_vol(prices=df_future["EUA Y0"], periods_per_year=num_pers)

In [15]:
_, _, _, _ = pricer.orchestrate_calculation(
    df_future=df_future, df_past=df_past, df_cal=df_calibration_data, 
    val_date=valuation_date, r=dicount_rate, num_periods=num_pers, 
    monthly_resampling=monthly_vol,
    price_correction_switch=price_cor_bool)

INFO - Pricing Strip Option....

INFO - Pricing European option(s) of strip...
INFO - European Option(S) of strip priced sucesfully!

INFO - Strip option priced sucesfully!!

INFO - Calcualting P&L from expired options of the strip...
INFO - P&L from expired options of the strip calculated sucesfully!!
